Create fbin bacteria mapping data frame

In [ ]:
import pandas as pd
from collections import Counter, defaultdict

map_df = pd.read_csv(f'/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/regression/segal_species/map_df.csv')
map_df

In [ ]:
mb_names = pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/mb_names.pkl")
mb_names

In [ ]:
# def rename_microbiome_columns(df: pd.DataFrame, mb_names: pd.DataFrame) -> pd.DataFrame:
#     """
#     Get either microbial_features (from diet_mb) or gut_bacteria_df (from the loader)
#     """
#     # --- Normalize ---
#     mb_names.index = mb_names.index.str.strip()
#     df = df.copy()
#     df.columns = df.columns.str.strip()

#     # --- Extract maps ---
#     species_map = mb_names['species_new'].str.strip()
#     genus_map = mb_names['genus_new'].str.strip()
#     family_map = mb_names['family_new'].str.strip()

#     # --- Build name mapping ---
#     final_mapping = {}
#     for col in df.columns[1:]:  # Skip RegistrationCode or first identifier column
#         name = species_map.get(col, None)
#         if name == "unknown" or pd.isna(name):
#             name = genus_map.get(col, None)
#         if name == "unknown" or pd.isna(name):
#             name = family_map.get(col, None)
#         if name is None or name == "unknown":
#             name = col  # fallback
#         final_mapping[col] = name

#     # --- Rename columns ---
#     df.rename(columns=final_mapping, inplace=True)

#     # --- Deduplicate ---
#     col_counts = Counter(df.columns)
#     name_counter = defaultdict(int)
#     new_cols = []

#     for col in df.columns:
#         if col_counts[col] > 1:
#             name_counter[col] += 1
#             new_cols.append(f"{col}_{name_counter[col]}")
#         else:
#             new_cols.append(col)

#     df.columns = new_cols

#     return df

# test = rename_microbiome_columns(mb_names, mb_names)

# test

In [ ]:
from collections import Counter, defaultdict
import pandas as pd

def rename_microbiome_columns_with_map(df: pd.DataFrame, mb_names: pd.DataFrame) -> (pd.DataFrame, dict):
    """
    Rename columns based on species/genus/family maps and return the renamed DataFrame
    along with a dictionary mapping original columns to their new names (with suffixes).
    """
    # --- Normalize ---
    mb_names = mb_names.copy()
    mb_names.index = mb_names.index.str.strip()
    df = df.copy()
    original_cols = df.columns.tolist()
    df.columns = df.columns.str.strip()

    # --- Extract maps ---
    species_map = mb_names['species_new'].str.strip()
    genus_map = mb_names['genus_new'].str.strip()
    family_map = mb_names['family_new'].str.strip()

    # --- Build initial mapping ---
    initial_map = {}
    for col in df.columns[1:]:  # Skip first identifier column
        name = species_map.get(col, None)
        if name == "unknown" or pd.isna(name):
            name = genus_map.get(col, None)
        if name == "unknown" or pd.isna(name):
            name = family_map.get(col, None)
        if name is None or name == "unknown":
            name = col  # fallback
        initial_map[col] = name

    # --- Rename columns ---
    df.rename(columns=initial_map, inplace=True)

    # --- Deduplicate and build final mapping ---
    col_counts = Counter(df.columns)
    name_counter = defaultdict(int)
    final_map = {}
    new_cols = []

    for orig, col in zip(original_cols, df.columns):
        # orig: original stripped col name; col: possibly mapped name
        if col_counts[col] > 1:
            name_counter[col] += 1
            new_name = f"{col}_{name_counter[col]}"
        else:
            new_name = col
        final_map[orig] = new_name
        new_cols.append(new_name)

    df.columns = new_cols
    return df, final_map

# Example usage:
renamed_df, mapping_dict = rename_microbiome_columns_with_map(mb, mb_names)
print(mapping_dict)


In [ ]:
map_df['daniel_species'] = 